In [1]:
import joblib
import pandas as pd

df_users = pd.read_parquet("data\\11092026\\users.parquet")
df_movies = pd.read_parquet("data\\11092026\\movies.parquet")
df_interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

encoders = joblib.load("data\\11092026\\encoders.pkl")
metadata = joblib.load("data\\11092026\\metadata.pkl")

In [2]:
df_users.head(10)

,user_id,gender,age_group,occupation,zip_code
0,1,0,1,10,48067
1,2,1,56,16,70072
2,3,1,25,15,55117
3,4,1,45,7,02460
4,5,1,25,20,55455
5,6,0,50,9,55117
6,7,1,35,1,06810
7,8,1,25,12,11413
8,9,1,25,17,61614
9,10,0,35,1,95370


In [3]:
df_movies.head(10)

,movie_id,title,genre,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,6,Heat (1995),Action|Crime|Thriller,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
6,7,Sabrina (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
7,8,Tom and Huck (1995),Adventure|Children's,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,9,Sudden Death (1995),Action,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,10,GoldenEye (1995),Action|Adventure|Thriller,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
df_interactions.head(10)

,user_idx,movie_idx,rating,timestamp
0,0,1104,5.0,978300760
1,0,639,3.0,978302109
2,0,853,3.0,978301968
3,0,3177,4.0,978300275
4,0,2162,5.0,978824291
5,0,1107,3.0,978302268
6,0,1195,5.0,978302039
7,0,2599,5.0,978300719
8,0,580,4.0,978302268
9,0,858,4.0,978301368


In [5]:
metadata

{'n_movies': 3706, 'n_users': 6040}

In [6]:
encoders

{'user_encoder': LabelEncoder(), 'movie_encoder': LabelEncoder()}

In [7]:
df_interactions["movie_id"] = (
    encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])
)

df_interactions["user_id"] = (
    encoders["user_encoder"].inverse_transform(df_interactions["user_idx"])
)

In [8]:
df_full = df_users.merge(df_interactions, on="user_id", how="left")
df_full = df_full.merge(df_movies, on="movie_id", how="left")

In [9]:
print(df_full.head(5))

   user_id  gender  age_group  occupation zip_code  user_idx  movie_idx  \
0        1       0          1          10    48067         0       1104   
1        1       0          1          10    48067         0        639   
2        1       0          1          10    48067         0        853   
3        1       0          1          10    48067         0       3177   
4        1       0          1          10    48067         0       2162   

   rating  timestamp  movie_id  ... Fantasy Film-Noir  Horror  Musical  \
0     5.0  978300760      1193  ...       0         0       0        0   
1     3.0  978302109       661  ...       0         0       0        1   
2     3.0  978301968       914  ...       0         0       0        1   
3     4.0  978300275      3408  ...       0         0       0        0   
4     5.0  978824291      2355  ...       0         0       0        0   

   Mystery  Romance  Sci-Fi  Thriller  War  Western  
0        0        0       0         0    0        

In [10]:
df_full.columns

Index(['user_id', 'gender', 'age_group', 'occupation', 'zip_code', 'user_idx',
       'movie_idx', 'rating', 'timestamp', 'movie_id', 'title', 'genre',
       'Action', 'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='str')

In [11]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit


In [56]:
drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "zip_code",
    "rating"
]

cat_features = [
    "user_id",
    "movie_id",
    "gender",
    "age_group",
    "occupation",
]

train , val , test = TemporalSplit().split(data=df_full)

group_train = train["user_id"]
group_val = val["user_id"]

y_train = train["rating"]
y_val = val["rating"]
y_test = test["rating"]

X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)

In [13]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(797758, 21)
(96719, 21)
(105732, 21)


In [57]:
from catboost import CatBoostRanker

model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=500,
    learning_rate=0.01,
    depth=5,
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    group_id=group_train,
    cat_features=cat_features
)

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 959ms	remaining: 7m 58s
50:	total: 42.8s	remaining: 6m 16s
100:	total: 1m 23s	remaining: 5m 30s
150:	total: 2m 4s	remaining: 4m 47s
200:	total: 2m 44s	remaining: 4m 4s
250:	total: 3m 24s	remaining: 3m 23s
300:	total: 4m 6s	remaining: 2m 43s
350:	total: 4m 49s	remaining: 2m 3s
400:	total: 5m 33s	remaining: 1m 22s
450:	total: 6m 16s	remaining: 40.9s
499:	total: 6m 59s	remaining: 0us


CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=500, learning_rate=0.01, loss_function='YetiRank', random_seed=42, verbose=50)

In [60]:
import numpy as np
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

drop_cols = [
    "genre",
    "title",
    "zip_code"
]

movies_ids = df_movies.movie_id.unique()

train_seen = (
    train
    .groupby("user_id")["movie_id"]
    .agg(set)
    .to_dict()
)
k=10
user_id = 1

watched = train_seen.get(user_id, set())

X = pd.DataFrame({
    "user_id": user_id,
    "movie_id": movies_ids
})

X = X[
    ~X["movie_id"].isin(watched)
]

X = X.merge(
    df_users,
    on="user_id",
    how="left"
)

X = X.merge(
    df_movies,
    on="movie_id",
    how="left"
)

movie_ids_recomendations = X["movie_id"].copy()

features = X.drop(
    columns=drop_cols
)

scores = model.predict(features)

top_indices = np.argsort(scores)[::-1][:k]

recomend = movie_ids_recomendations.iloc[
    top_indices
]

val_per_user = val[
    val["user_id"] == user_id
]

relevant = val_per_user.loc[
    val["rating"] >= 4,
    "movie_id"
]

recall = recall_at_k(relevant, recomend)
precision = precision_at_k(relevant, recomend)
ndcg = ndcg_at_k(relevant, recomend)

print(recall)
print(precision)
print(ndcg)

0.0
0.0
0.0


In [61]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)
import numpy as np


def eval(
        model,
        train,
        val,
        df_movie,
        df_users,
        df_interactions,
        k=10,
):

    movies_ids = df_movie.movie_id.unique()

    train_seen = (
        train
        .groupby("user_id")["movie_id"]
        .agg(set)
        .to_dict()
    )

    recalls = []
    precisions = []
    ndcgs = []

    for user_id in val.user_id.unique():
        watched = train_seen.get(user_id, set())

        X = pd.DataFrame({
            "user_id": user_id,
            "movie_id": movies_ids
        })

        X = X[
            ~X["movie_id"].isin(watched)
        ]

        X = X.merge(
            df_users,
            on="user_id",
            how="left"
        )

        X = X.merge(
            df_movie,
            on="movie_id",
            how="left"
        )

        movie_ids_recomendations = X["movie_id"].copy()

        features = X.drop(
            columns=drop_cols
        )

        scores = model.predict(features)

        top_indices = np.argsort(scores)[::-1][:k]

        recomend = movie_ids_recomendations.iloc[
            top_indices
        ]

        val_per_user = val[
            val["user_id"] == user_id
        ]

        relevant = val_per_user.loc[
            val["rating"] >= 4,
            "movie_id"
        ]

        recall = recall_at_k(relevant, recomend)
        precision = precision_at_k(relevant, recomend)
        ndcg = ndcg_at_k(relevant, recomend)

        recalls.append(recall)
        precisions.append(precision)
        ndcgs.append(ndcg)

    return {
        "recall": np.mean(recalls),
        "precision": np.mean(precisions),
        "ndcg": np.mean(ndcgs)
    }




    print(train_seen)


In [62]:
res = eval(model, train, val, df_movies, df_users, df_interactions)

Первая версмия модели с пользователями и контентными признаками показала нозкую качество модели. После добавления user_id и movie_id качество модели на validation метриках возросло, но это от части основано на запоминания пользователей и фильмов (хотя даже так вышли не самые лучшие метрики). Поэтому далее я попробую исмпользховать двухэтапную систему рекомендаций: Popularity + Matrix Factorization, после чего CatBoost будет выполнять ранжирования кандидатов. Так же будут добавленны новые признаки: Факторы MF и оценка MF. (возможно будут так же добавленны новые признаки на основе дданных с таблиц)

'recall': np.float64(0.026596517016002793), \
 'precision': np.float64(0.022367549668874175), \
 'ndcg': np.float64(0.00024488983035845813)

In [63]:
res

{'recall': np.float64(0.026596517016002793),
 'precision': np.float64(0.022367549668874175),
 'ndcg': np.float64(0.00024488983035845813)}